In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.io.shapereader as shpreader
import pycountry

In [ ]:
from _helpers_notebooks import mock_snakemake

snakemake = mock_snakemake(
    "plot_regions",
)

### Get regions from config

In [ ]:
config = snakemake.config

In [ ]:
regions = config["regions"]

In [ ]:
# Regions in config already use ISO alpha_3 codes — no name corrections needed

### Get global map

In [ ]:
# Use cartopy to get natural earth countries shapefile
shapename = "admin_0_countries"
reader = shpreader.natural_earth(resolution="110m", category="cultural", name=shapename)

world = gpd.read_file(reader)

### Merge global dataset with regions from config

In [ ]:
def admin_to_iso_a3(country_name):
    try:
        return pycountry.countries.lookup(country_name).alpha_3
    except LookupError:
        return None

In [ ]:
# Build a mapping from ISO_A3 code to region directly from config
iso_to_region = {}
for region, countries in regions.items():
    for iso3 in countries:
        iso_to_region[iso3] = region

# Fix missing ISO_A3 codes in world (some entries have -99)
world.loc[world["ISO_A3"] == "-99", "ISO_A3"] = world.loc[
    world["ISO_A3"] == "-99", "ADMIN"
].apply(admin_to_iso_a3)

In [ ]:
world["region"] = world["ISO_A3"].map(lambda iso: iso_to_region.get(iso, "Other"))

### Plot global map

In [ ]:
# Short mapping: assume config['colors'] is a dict region->color
import matplotlib.patches as mpatches

color_map = config.get("colors", {})

world["plot_color"] = world["region"].map(color_map).fillna("lightgrey")

fig, ax = plt.subplots(figsize=(12, 10))

world.plot(ax=ax, color=world["plot_color"], edgecolor="white", alpha=0.8)

# Manual legend for regions present in the map
handles = [
    mpatches.Patch(color=c, label=r, alpha=0.8)
    for r, c in color_map.items()
    if r in world["region"].values
]
if handles:
    region_list = [r for r, c in color_map.items() if r in world["region"].values]
    labels = [reg.replace("_", " ") for reg in region_list]
    ax.legend(
        handles=handles, labels=labels, title="Region", bbox_to_anchor=(1.05, 1), loc="upper left",
    )

# ax.set_title('Regions')
ax.axis("off")
plt.tight_layout()

plt.savefig(snakemake.output.global_map_countries)
plt.savefig(snakemake.output.global_map_countries_png, dpi=300)
plt.show()